In [1]:
import re
import math
from collections import defaultdict

In [2]:
STOP_WORDS = {'a','an','the','is','of','for','and','or','was','are'}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    words = text.split()
    return [w.rstrip('s') for w in words if w not in STOP_WORDS]

In [3]:
class PageEntry:
    def __init__(self, name, text):
        self.name = name
        self.words = preprocess(text)
        self.index = defaultdict(list)
        for i, word in enumerate(self.words):
            self.index[word].append(i)

In [4]:
class InvertedIndex:
    def __init__(self):
        self.index = defaultdict(list)
        self.pages = []
    def addPage(self, page):
        self.pages.append(page)
        for word, positions in page.index.items():
            self.index[word].append((page.name, positions))
    def search(self, word):
        return self.index.get(word, [])

In [5]:
def tf(word, page):
    return page.words.count(word) / len(page.words)

def idf(word, pages):
    N = len(pages)
    count = sum(1 for p in pages if word in p.words)
    return math.log(N / (count if count else 1))

def tfidf(word, page, pages):
    return tf(word, page) * idf(word, pages)

In [11]:
print("-----------------------------------------------------------------------------------------------------------------------------------------------------------")
print('SEARCH ENGINE OUTPUT')
print("-----------------------------------------------------------------------------------------------------------------------------------------------------------")

files = [
    'stack_datastructure_wiki',
    'stack_cprogramming',
    'stack_oracle',
    'stackoverflow',
    'stacklighting',
    'stackmagazine'
]

index = InvertedIndex()

for fname in files:
    with open(fname, 'r', encoding='utf-8') as f:
        text = f.read()
        page = PageEntry(fname, text)
        index.addPage(page)

print('\nQuery: Stack\n')
for item in index.search("stack"):
    print(item)
print("\n")
print("-----------------------------------------------------------------------------------------------------------------------------------------------------------")
print('\nQuery: Data\n')
for item in index.search("data"):
    print(item)
print("\n")
print("-----------------------------------------------------------------------------------------------------------------------------------------------------------")
print('\nTF-IDF (data in stack_datastructure_wiki):\n')
for p in index.pages:
    if p.name == 'stack_datastructure_wiki':
        print(tfidf('data', p, index.pages))
print("\n")
print("-----------------------------------------------------------------------------------------------------------------------------------------------------------")

-----------------------------------------------------------------------------------------------------------------------------------------------------------
SEARCH ENGINE OUTPUT
-----------------------------------------------------------------------------------------------------------------------------------------------------------

Query: Stack

('stack_datastructure_wiki', [0, 11, 55, 96, 105, 114, 127, 141, 151, 154, 167, 171, 190, 198, 214, 217, 240, 249, 258, 268, 284, 293, 304, 324, 328, 342])
('stack_cprogramming', [0, 9, 69, 85, 105, 116, 118, 135, 142, 150, 159, 173, 187])
('stack_oracle', [7, 11, 16, 24, 40, 56, 61, 66, 75, 87, 107])
('stackoverflow', [0])
('stacklighting', [1, 6, 12])
('stackmagazine', [0, 20])


-----------------------------------------------------------------------------------------------------------------------------------------------------------

Query: Data

('stack_datastructure_wiki', [2, 18, 77, 173, 315])
('stack_cprogramming', [1, 11, 77, 87])
('sta